# Revisi - Tahap 3: Penilaian Kualitas Data (Objektif)

Dua metode objektif yang saling lengkapi:

- **A. Heuristik** (deterministik, CPU, bisa diulang persis) - `inspect_data` + grader eval.
  Mengukur konsistensi internal & kerusakan format secara terukur.
- **B. LLM-as-judge** (model nilai sendiri) - `judge_data_quality`. Mengukur kebenaran
  matematis gold/solusi tanpa bergantung pada label dataset.

Heuristik = murah & pasti. LLM-judge = mendekati "kebenaran" tapi dibatasi kekuatan judge.
Dipakai bareng -> kalau keduanya sepakat, sinyalnya kuat.

In [1]:
import os, sys
p = os.getcwd()
while not os.path.isdir(os.path.join(p, 'src')) and os.path.dirname(p) != p:
    p = os.path.dirname(p)
os.chdir(p); sys.path.insert(0, p)
import pandas as pd
from IPython.display import Markdown, display
print('repo root:', p)

repo root: D:\Main Storage\Vscode\FP_NLP\FP_NLP


In [2]:
SETS = {
    'test_numglue': 'data/sft/test/numglue_test.jsonl',
    'test_easy':    'data/sft/test/easy_test.jsonl',
    'train_cot':    'data/sft/train/cot.jsonl',
    'train_nocot':  'data/sft/train/nocot.jsonl',
}

## Apa yang dicek & dinilai

In [3]:
from src.eval.data_quality_report import checks_table_md
display(Markdown(checks_table_md()))

| metrik | yang dicek | metode | baik bila |
| --- | --- | --- | --- |
| `self_consistency` | cara dataset == gold (plafon akurasi) | heuristik(grader) | tinggi |
| `gold_benar_rate` | gold benar secara matematis | LLM-judge | tinggi |
| `soal_lengkap_rate` | soal self-contained (tak butuh data luar) | LLM-judge | tinggi |
| `unanswerable_rate` | soal nyebut tabel/teks luar tak disertakan | heuristik(regex) | rendah |
| `empty_jawaban_rate` | gold kosong | heuristik | rendah |
| `text_answer_rate` | gold kalimat-bebas (susah auto-grade) | heuristik | rendah |
| `mojibake_rate` | teks rusak (encoding) | heuristik(regex) | rendah |
| `dup_rate` | soal duplikat | heuristik | rendah |
| `jawaban_benar_rate` | (train) jawaban solusi benar | LLM-judge | tinggi |
| `penalaran_valid_rate` | (train) langkah valid, bukan ngarang | LLM-judge | tinggi |
| `tanpa_boxed_rate` | (train) solusi tanpa \boxed{} | heuristik | rendah |

## Metode A - Heuristik (deterministik)

`self_consistency` rendah = label noise (plafon akurasi). `unanswerable`/`mojibake`/
`text_answer`/`empty` tinggi = data rusak. Train: `tanpa_boxed`/`format_issue` tinggi = SFT cacat.

In [4]:
from src.eval.data_quality_report import heuristic_metrics
rows = [heuristic_metrics(p) for p in SETS.values()]
gold = pd.DataFrame([r for r in rows if r['kind']=='gold']).set_index('dataset')
train = pd.DataFrame([r for r in rows if str(r['kind']).startswith('chatml')]).set_index('dataset')
print('TEST (gold):'); display(gold)
print('TRAIN (chatml):'); display(train)

TEST (gold):


,kind,n,self_consistency,unanswerable_rate,empty_jawaban_rate,text_answer_rate,mojibake_rate,dup_rate
dataset,,,,,,,,
numglue_test,gold,300,0.537,0.003,0.0,0.000,0.00,0.0
easy_test,gold,300,0.327,0.010,0.0,0.253,0.01,0.0


TRAIN (chatml):


,kind,n,tanpa_boxed_rate,mojibake_rate,dup_rate,format_issue_rate
dataset,,,,,,
cot,chatml/cot,2709,0.0,0.002,0.0,0.0
nocot,chatml/nocot,2709,0.0,0.001,0.0,0.0


## Metode B - LLM-as-judge (objektif)

Judge (Groq llama-3.3-70b) menyelesaikan/verifikasi sendiri. `gold_benar`/`jawaban_benar`
= yakin benar; sisanya bisa `ragu` (judge tak yakin / soal butuh data luar). Tabel ini
baca verdict dari `data/judge/` (hasil `python -m src.eval.judge_data_quality`).

In [5]:
from src.eval.data_quality_report import judge_metrics
jr = []
for name, p in SETS.items():
    import os
    stem = os.path.splitext(os.path.basename(p))[0]
    jm = judge_metrics(stem)
    if jm: jm = {'dataset': name, **jm}; jr.append(jm)
if jr:
    display(pd.DataFrame(jr).set_index('dataset'))
else:
    print('Belum ada verdict. Jalankan: python -m src.eval.judge_data_quality --sample 80')

,kind,n,gold_benar_rate,gold_salah_rate,gold_ragu_rate,soal_lengkap_rate,jawaban_benar_rate,jawaban_salah_rate,penalaran_valid_rate,penalaran_cacat_rate
dataset,,,,,,,,,,
test_numglue,gold,12,0.417,0.25,0.333,0.417,NaN,NaN,NaN,NaN
test_easy,gold,12,0.750,0.00,0.250,0.833,NaN,NaN,NaN,NaN
train_cot,train,12,NaN,NaN,NaN,NaN,0.917,0.083,0.917,0.083
train_nocot,train,12,NaN,NaN,NaN,NaN,0.750,0.000,0.667,0.000


### Contoh gold yang divonis SALAH oleh judge (bukti objektif label noise)

In [6]:
import json, os
fp = 'data/judge/numglue_test.verdicts.jsonl'
if os.path.exists(fp):
    v = [json.loads(l) for l in open(fp, encoding='utf-8')]
    bad = [r for r in v if r.get('gold')=='salah'][:4]
    for r in bad:
        print('gold =', r.get('gold_ans'), '| soal:', r['soal'][:90])
    if not bad: print('(tak ada yang divonis salah di sampel ini)')
else:
    print('verdict numglue belum ada')

gold = 15 | soal: Berapa poin persentase PDB utang publik meningkat antara tahun 1980 dan 1988?
gold = 1 | soal: Berapa hari setelah pendaratan pasukan Unit 124 memasuki desa-desa?
gold = 2 | soal: Berapa banyak negara di tur musim panas 2014 yang dijadwalkan untuk lebih dari satu tangga


## Kesimpulan

Bandingkan **TEST gold** vs **TRAIN**:
- TEST gold: `self_consistency` rendah (numglue ~54%, easy ~33%) + `gold_benar` (judge) rendah
  -> banyak label test tak bisa dipercaya. numglue: banyak soal butuh data luar (ragu/unanswerable).
- TRAIN: `tanpa_boxed`/`mojibake` ~0 + `jawaban_benar` (judge) jauh lebih tinggi -> data SFT relatif bersih
  (rejection sampling nyaring gold salah saat training).

**Implikasi:** pass@ rendah sebagian besar = test gold bermasalah + model kecil, BUKAN train rusak.

### Caveat (jujur)
- LLM-judge dibatasi kekuatan model: soal yang butuh tabel/data luar -> `ragu` (bukan berarti gold benar/salah).
- Sampel kecil (cepat/murah). Naikkan `--sample` + model lebih kuat (`--model`) untuk angka lebih stabil.
- Heuristik = sinyal pasti; LLM-judge = pelengkap. Pakai dua-duanya.